In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [4]:
!pip install gensim wandb wikipedia-api langchain langchain_text_splitters langchain-community langchain-huggingface faiss-cpu transformers accelerate bitsandbytes --quiet
!pip install --upgrade transformers huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.2/129.2 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 41.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 77.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 45.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 50.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.7/561.7 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is no

In [5]:
import os
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from gensim.models import Word2Vec
import wandb
 
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
 
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

Using device: cuda


In [6]:
train_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
 
print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)
print("\nMissing values in train:\n", train_df.isnull().sum())
print("\nAnswer label distribution:\n", train_df["answer"].value_counts())
 
train_df["prompt_len"] = train_df["prompt"].astype(str).apply(lambda x: len(x.split()))
print("\nPrompt word-length stats:\n", train_df["prompt_len"].describe())

print(train_df.head(3))

Train shape: (2000, 8)
Test shape : (500, 7)

Missing values in train:
 id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
answer    0
dtype: int64

Answer label distribution:
 answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64

Prompt word-length stats:
 count    2000.00000
mean       18.14650
std         6.78189
min         3.00000
25%        14.00000
50%        17.00000
75%        22.00000
max        51.00000
Name: prompt_len, dtype: float64
   id                                             prompt  \
0   1  Pick the best possible answer: What is Martin ...   
1   2        What is accelerator-based light-ion fusion?   
2   3  Determine the correct option: What is the term...   

                                                   A  \
0  Martin Heidegger believes that humans exist wi...   
1  Accelerator-based light-ion fusion is a techni...   
2                                       Blueshifting   

                         

In [7]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text
 
 
TEXT_COLS = ["prompt", "A", "B", "C", "D", "E"]
 
for col in TEXT_COLS:
    train_df[col] = train_df[col].apply(clean_text)
    test_df[col] = test_df[col].apply(clean_text)

def tokenize(text):
    return text.split()

In [9]:
all_sentences = []
for df in [train_df, test_df]:
    for col in TEXT_COLS:
        all_sentences.extend(df[col].apply(tokenize).tolist())
 
EMBED_DIM = 100
 
w2v_model = Word2Vec(
    sentences=all_sentences,
    vector_size=EMBED_DIM,
    window=5,
    min_count=1,
    workers=4,
    sg=1,
    seed=SEED,
)
 
print("Vocabulary size:", len(w2v_model.wv))
w2v_model.save(os.path.join("/kaggle/working", "word2vec.model"))

Vocabulary size: 2973


In [10]:
def text_to_vector(text, model, dim=EMBED_DIM):
    words = tokenize(text)
    vecs = []

    for word in words:
        if word in model.wv:
            vecs.append(model.wv[word])

    if len(vecs) == 0:
        return np.zeros(dim, dtype=np.float32)

    avg_vector = np.mean(vecs, axis=0)
    return avg_vector.astype(np.float32)

In [11]:
LABELS = ["A", "B", "C", "D", "E"]
LABEL2IDX = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}

class MCQDataset(Dataset):

    def __init__(self, df, w2v_model, has_labels=True):
        self.df = df.reset_index(drop=True)
        self.model = w2v_model
        self.has_labels = has_labels

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        question_vector = text_to_vector(row["prompt"], self.model)
        
        features = []
        for option in LABELS:
            option_vector = text_to_vector(row[option], self.model)
            feature = np.concatenate((question_vector, option_vector))
            features.append(feature)

        data = {}
        data["features"] = torch.tensor(features, dtype=torch.float32)

        if self.has_labels:
            answer = LABEL2IDX[row["answer"]]
            data["label"] = torch.tensor(answer, dtype=torch.long)
        else:
            data["id"] = row["id"]

        return data

In [12]:
class MCQScorer(nn.Module):
    def __init__(self, input_dim, hidden_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )
 
    def forward(self, x):
        batch, n_options, dim = x.shape
        x = x.view(batch * n_options, dim)
        scores = self.net(x)
        scores = scores.view(batch, n_options)
        return scores

In [13]:
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
except:
    pass

wandb.login()

# wandb.init(
#     project="dlgenai-project-26t2",
#     name="from-scratch-nn",
#     config={
#         "model": "from_scratch_nn",
#         "batch_size": 64,
#         "epochs": 50,
#         "lr": 1e-3,
#         "hidden_dim": 256
#     }
# )

# cfg = wandb.config

train_data, val_data = train_test_split(
    train_df,
    test_size=0.20,
    random_state=SEED,
    stratify=train_df["answer"]
)

# train_loader = DataLoader(
#     MCQDataset(train_data, w2v_model),
#     batch_size=cfg.batch_size,
#     shuffle=True
# )

# val_loader = DataLoader(
#     MCQDataset(val_data, w2v_model),
#     batch_size=cfg.batch_size,
#     shuffle=False
# )

# model = MCQScorer(2 * EMBED_DIM, cfg.hidden_dim).to(DEVICE)

# criterion = nn.CrossEntropyLoss()
# optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr)

# best_val_f1 = 0

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: 23f2003771 (qubyt) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [ ]:
# from sklearn.metrics import accuracy_score, f1_score

# def run_one_epoch(model, loader, optimizer=None):
    
#     is_training = optimizer is not None

#     if is_training:
#         model.train()
#     else:
#         model.eval()

#     total_loss = 0
#     all_preds = []
#     all_labels = []

#     for batch in loader:

#         x = batch["features"].to(DEVICE)
#         y = batch["label"].to(DEVICE)

#         if is_training:
#             output = model(x)
#             loss = criterion(output, y)

#             optimizer.zero_grad()
#             loss.backward()
#             optimizer.step()
#         else:
#             with torch.no_grad():
#                 output = model(x)
#                 loss = criterion(output, y)

#         total_loss += loss.item() * x.size(0)

#         preds = output.argmax(dim=1)
#         all_preds.append(preds.detach().cpu().numpy())
#         all_labels.append(y.cpu().numpy())

#     all_preds = np.concatenate(all_preds)
#     all_labels = np.concatenate(all_labels)

#     avg_loss = total_loss / len(loader.dataset)
#     acc = accuracy_score(all_labels, all_preds)
#     f1 = f1_score(all_labels, all_preds, average="weighted")

#     return avg_loss, acc, f1


# for epoch in range(cfg.epochs):

#     train_loss, train_acc, train_f1 = run_one_epoch(model, train_loader, optimizer)
#     val_loss, val_acc, val_f1 = run_one_epoch(model, val_loader, optimizer=None)

#     wandb.log({
#         "train_loss": train_loss,
#         "train_accuracy": train_acc,
#         "train_f1": train_f1,
#         "val_loss": val_loss,
#         "val_accuracy": val_acc,
#         "val_f1": val_f1
#     })

#     print(
#         f"Epoch {epoch+1}/{cfg.epochs} | "
#         f"Train F1: {train_f1:.4f} | "
#         f"Val F1: {val_f1:.4f}"
#     )

#     if val_f1 > best_val_f1:
#         best_val_f1 = val_f1
#         torch.save(model.state_dict(), "best_model.pt")

# wandb.finish()

In [ ]:
# model = MCQScorer(2 * EMBED_DIM, 256).to(DEVICE)
# model.load_state_dict(torch.load("best_model.pt", map_location=DEVICE))
# model.eval()

# test_loader = DataLoader(
#     MCQDataset(test_df, w2v_model, has_labels=False),
#     batch_size=64,
#     shuffle=False
# )

# predictions = []

# with torch.no_grad():

#     for batch in test_loader:

#         x = batch["features"].to(DEVICE)

#         probs = torch.softmax(model(x), dim=1).cpu().numpy()
#         top3 = np.argsort(-probs, axis=1)[:, :3]

#         for pred in top3:
#             pred_letters = []
#             for i in pred:
#                 pred_letters.append(LABELS[i])
#             predictions.append(" ".join(pred_letters))

# test_df["Prediction_Model_NN"] = predictions

# print("Predictions generated successfully.")

In [14]:
wiki_topics = [
    "Supersymmetric quantum mechanics", "Heisenberg uncertainty principle", "Virtual particle",
    "Spontaneous symmetry breaking", "Wigner distribution function", "Magnetic monopole",
    "Spin quantum number", "Parity (physics)", "Peierls bracket", "Geometric quantization",
    "Quantum field theory", "Hilbert space", "Probability amplitude", "Ramsauer–Townsend effect",
    "Explicit symmetry breaking", "Angular momentum operator", "Standard Model",
    "Higgs boson", "CP violation", "Quark", "Chemical potential",
    "Lorentz covariance", "Minkowski space", "Minkowski diagram", "Special relativity",
    "General relativity", "Simultaneity", "Speed of light", "Born reciprocity",
    "Frame-dragging", "Gravitomagnetism", "Gravity Probe B", "Roche limit",
    "Penrose process", "Black hole information paradox", "Schwarzschild black hole",
    "CEERS-93316", "James Webb Space Telescope", "Redshift", "Metric expansion of space",
    "Proper distance", "Interstellar medium", "Molecular cloud", "Supernova remnant",
    "Main sequence", "Pulsar", "Crab Pulsar", "Supermassive black hole",
    "Sagittarius A*", "Dark matter", "Gravitational wave", "Doppler effect",
    "Lyman-alpha line", "Planetary system", "X-ray pulsar-based navigation",
    "Baryon acoustic oscillations", "Modified Newtonian dynamics", "Inflaton",
    "Einstein@Home", "Light-year", "Apparent magnitude", "Metallicity",
    "Kapteyn's Star", "Isophote", "Type Ia supernova", "Supernova",
    "Carnot heat engine", "Maxwell's demon", "Throttling process", "Second law of thermodynamics",
    "Kelvin–Helmholtz instability", "Coherent turbulent structure", "Cavitation", "Convection",
    "Natural convection", "Bernoulli's principle", "Kutta condition", "Navier–Stokes equations",
    "Cauchy momentum equation", "Water hammer",
    "Fermat's principle", "Emissivity", "Illuminance", "Luminance", "Rayleigh scattering",
    "Young's interference experiment", "Diffraction", "Total internal reflection",
    "Radiosity (radiometry)", "Stefan–Boltzmann law", "Ultraviolet catastrophe",
    "Optical signal-to-noise ratio", "Propagation constant", "Loudness",
    "Landau–Lifshitz–Gilbert equation", "Magnetic susceptibility", "Memristor",
    "Spin valve", "Electrical resistivity and conductivity", "Superconductivity",
    "Amorphous metal", "Variable-range hopping", "Piezoelectricity", "Dielectric loss",
    "Josephson effect", "De Haas–Van Alphen effect", "Paramagnetism", "Order parameter",
    "Ferroelectricity", "ReRAM", "Evans balance", "Spatial dispersion",
    "Identity element", "Crystallographic point group", "Improper rotation",
    "Crystallinity", "API gravity", "Radiometric dating", "Recrystallization (metallurgy)", "Grain boundary strengthening", 
    "Fischer–Tropsch process", "Carbocation", "Naphthalene", "Crossover experiment",
    "Fourier-transform infrared spectroscopy", "Three moment theorem", "Bollard pull", "Ring-imaging Cherenkov detector", 
    "Formal system", "Uniform tilings in hyperbolic plane", "Regular polytope",
    "Probability density function", "Probability mass function", "Reciprocal length",
    "Symmetry group", "Erlangen program", "Hyperbolic geometry", "Permutation group",
    "CW complex", "Dimension", "Hesse's principle of transfer", "Dynamic scaling", "Liouville's theorem (Hamiltonian)", 
    "Surgical pathology", "Active transport", "Trophic level", "Pulmonary circulation",
    "Mammary gland", "Organography", "Cyclotide", "Phageome",
    "Myrmecophyte", "Mycorrhiza", "IL-10", "Regulatory T cell", "Anatomy", "Cardiac skeleton",
    "Second", "Coordinated Universal Time", "Universal Time",
    "Triskelion", "Newton's laws of motion", "Right-hand rule", "Giordano Bruno",
    "Shower-curtain effect", "Wilson cloud chamber", "Ozma Problem", "Horror vacui",
    "Butterfly effect", "Gauss's law", "Scale (map)", "Martin Heidegger",
    "Isaac Newton", "Robert Hooke", "Pierre de Fermat",
    "Classical mechanics", "Earnshaw's theorem", "Environmental Science Center",
    "Memristor", "Synaptic transistor", "Power density", "Cold dark matter", "Antimatter",
    "Baryon asymmetry", "L dwarf",
    "Pycnometer", "Photophoresis", "Isophote", "Recycling", "Rare-earth element",
    "Fusor", "Thylakoid", "Diquark", "Thermodynamic system", "Molecular symmetry",
    "Mass-to-charge ratio", "Rømer's determination of the speed of light",
    "Resistive random-access memory", "Diffuse sky radiation", "Grain growth",
]

In [23]:
import os
import wikipediaapi
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

wiki = wikipediaapi.Wikipedia(
    user_agent="MyRAGProject/1.0 (singhshikhar8957@gmail.com)",
    language="en"
)

print("Scraping Wikipedia to build the Knowledge Base...")

scraped_texts = []

for topic in wiki_topics:
    try:
        page = wiki.page(topic)
        if page.exists():
            scraped_texts.append(page.text)
        else:
            print(f"Skipped {topic}: page not found")
    except Exception as e:
        print(f"Skipped {topic} due to error: {e}")
print(f"Successfully scraped {len(scraped_texts)} articles.")

text_splitter = RecursiveCharacterTextSplitter(chunk_size=900, chunk_overlap=300)
docs = text_splitter.create_documents(scraped_texts)
print(f"Created {len(docs)} chunks.")

embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-large-en-v1.5", 
                                   model_kwargs={"device": DEVICE,
                                                "model_kwargs": {"use_safetensors": False}},
                                   encode_kwargs={"normalize_embeddings": True}
)

vector_db = FAISS.from_documents(docs, embeddings)
print("FAISS index created successfully.")

Scraping Wikipedia to build the Knowledge Base...
Successfully scraped 200 articles.
Created 11492 chunks.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

FAISS index created successfully.


In [16]:
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline, logging
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
import torch

transformers.logging.set_verbosity_error()

try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    login(hf_token)
    print("Logged into Hugging Face successfully!")
except Exception as e:
    print(f"HF login failed: {e}. Please ensure you have added a 'HF_TOKEN' secret in Kaggle.")

model_id = "google/gemma-4-12B"

print(f"Loading {model_id} from Hugging Face...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.padding_side = "left"

llm_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

llm_pipe = pipeline(
    "text-generation", 
    model=llm_model, 
    tokenizer=tokenizer, 
    max_new_tokens=20, 
    do_sample=False, 
    return_full_text=False, 
    pad_token_id=tokenizer.eos_token_id
)
print("LLM loaded successfully and ready for inference!")

Logged into Hugging Face successfully!
Loading google/gemma-4-12B from Hugging Face...


config.json:   0%|          | 0.00/4.38k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/888 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 32.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B / 23.9GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/677 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/233 [00:00<?, ?B/s]

LLM loaded successfully and ready for inference!


In [24]:
import pandas as pd
from langchain_core.documents import Document

print("Building Train Dataset Vector DB...")
train_docs = []

for _, row in train_df.iterrows():
    train_text = (
        f"Question: {row['prompt']}\n"
        f"A: {row['A']}\nB: {row['B']}\nC: {row['C']}\nD: {row['D']}\nE: {row['E']}\n"
        f"Correct Answer: {row['answer']}"
    )
    doc = Document(page_content=train_text)
    train_docs.append(doc)

train_vector_db = FAISS.from_documents(train_docs, embeddings)
print("Train Dataset Vector DB created successfully!")

Building Train Dataset Vector DB...
Train Dataset Vector DB created successfully!


In [20]:
def retrieve_context(question, k=3):
    docs = vector_db.similarity_search(question, k=k)
    context_parts = []
    for d in docs:
        context_parts.append(d.page_content)
    return "\n".join(context_parts)

def retrieve_train_examples(question, k=2):
    docs = train_vector_db.similarity_search(question, k=k)
    context_parts = []
    for d in docs:
        context_parts.append(d.page_content)
    return "\n\n".join(context_parts)

def build_prompt(question, options, wiki_context, train_examples):
    options_text = ""
    for label in ["A", "B", "C", "D", "E"]:
        if label in options:
            options_text += label + ": " + options[label] + "\n"
            
    prompt = (
        "Wiki Context:\n" + wiki_context + "\n\n"
        "Similar Examples from Training Data (Pay close attention to the Correct Answer here):\n" + train_examples + "\n\n"
        "Question: " + question + "\n\n" + options_text + "\n"
        "Based on the similar examples and Wiki context above, rank the 3 most likely correct options.\n"
        "Reply with exactly 3 letters separated by spaces, nothing else.\n"
        "Answer:"
    )
    return prompt

def generate_answer(prompt):
    return llm_pipe(prompt, max_new_tokens=10)[0]['generated_text']

def parse_letters(raw_output, valid_labels):
    found = re.findall(r'\b[A-E]\b', raw_output.upper())
    letters = []
    for ch in found:
        if ch in valid_labels and ch not in letters:
            letters.append(ch)
    return letters

def normalize(text):
    text = text.lower()
    return re.sub(r"[^a-z0-9\s]", " ", text)

def token_overlap_score(option_text, context_text):
    opt_tokens = set(normalize(option_text).split())
    ctx_tokens = set(normalize(context_text).split())
    if not opt_tokens:
        return 0.0
    return len(opt_tokens & ctx_tokens) / len(opt_tokens)

def similarity_backup_ranking(options, context):
    scores = {}
    for label, text in options.items():
        scores[label] = token_overlap_score(text, context)
    return sorted(scores, key=scores.get, reverse=True)

def get_row_options(row):
    options = {}
    for k in ["A", "B", "C", "D", "E"]:
        if k in row and str(row[k]) != "nan":
            options[k] = str(row[k])
    return options

def finalize_letters(raw_output, options, wiki_context):
    letters = parse_letters(raw_output, list(options.keys()))
    backup = similarity_backup_ranking(options, wiki_context)

    for label in backup:
        if len(letters) >= 3:
            break
        if label not in letters:
            letters.append(label)

    return " ".join(letters[:3])

def answer_question(row):
    question = row["prompt"]
    options = get_row_options(row)

    wiki_context = retrieve_context(question)
    train_examples = retrieve_train_examples(question, k=2)
    prompt = build_prompt(question, options, wiki_context, train_examples)
    raw_output = generate_answer(prompt)

    return finalize_letters(raw_output, options, wiki_context)

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

wandb.init(
    project="dlgenai-project-26t2",
    name="rag-pipeline",
    config={"model": "pretrained_rag_llm", "llm": model_id, "k_wiki": 3, "k_train_examples": 2}
)

val_data_sample = val_data.sample(50, random_state=SEED)

val_answers = [answer_question(row) for _, row in val_data_sample.iterrows()]

val_preds = []
for answer in val_answers:
    pred_list = answer.split()
    if pred_list:
        val_preds.append(pred_list[0])
    else:
        val_preds.append(None)

val_true = []
for answer in val_data_sample["answer"]:
    val_true.append(answer)

rag_val_acc = accuracy_score(val_true, val_preds)
rag_val_f1 = f1_score(val_true, val_preds, average="weighted", labels=LABELS)

wandb.log({"val_accuracy": rag_val_acc, "val_f1": rag_val_f1})
print(f"RAG Val Accuracy: {rag_val_acc:.4f} | RAG Val F1: {rag_val_f1:.4f}")

test_predictions = [answer_question(row) for _, row in test_df.iterrows()]
test_df["Prediction_Model_Rag"] = test_predictions

wandb.finish()

In [ ]:
# from sklearn.linear_model import LogisticRegression

# def create_lr_features(df, w2v_model):
#     X = []
#     for _, row in df.iterrows():
#         q_vec = text_to_vector(row["prompt"], w2v_model)
        
#         opt_vecs = []
#         for opt in ["A", "B", "C", "D", "E"]:
#             text_data = row[opt]
#             vector = text_to_vector(text_data, w2v_model)
#             opt_vecs.append(vector)
            
#         feature_row = np.concatenate([q_vec] + opt_vecs)
#         X.append(feature_row)
        
#     return np.array(X)

In [ ]:
# from sklearn.metrics import accuracy_score, f1_score

# X_train_lr = create_lr_features(train_data, w2v_model)
# reverse_map = {0: "A", 1: "B", 2: "C", 3: "D", 4: "E"}

# y_train_lr_list = []
# for ans in train_data["answer"]:
#     numeric_label = LABEL2IDX[ans]
#     y_train_lr_list.append(numeric_label)
# y_train_lr = np.array(y_train_lr_list)

# wandb.init(
#     project="dlgenai-project-26t2",
#     name="logistic-regression",
#     config={"model": "logistic_regression", "max_iter": 5000}
# )

# print("Training Logistic Regression Model...")
# lr_model = LogisticRegression(C=0.3, penalty='l2', max_iter=1000, random_state=SEED)
# lr_model.fit(X_train_lr, y_train_lr)

# train_preds_lr = lr_model.predict(X_train_lr)
# train_acc_lr = accuracy_score(y_train_lr, train_preds_lr)
# train_f1_lr = f1_score(y_train_lr, train_preds_lr, average="weighted")

# X_val_lr = create_lr_features(val_data, w2v_model)

# y_val_lr_list = []
# for ans in val_data["answer"]:
#     numeric_label = LABEL2IDX[ans]
#     y_val_lr_list.append(numeric_label)
# y_val_lr = np.array(y_val_lr_list)

# val_preds_lr = lr_model.predict(X_val_lr)
# val_acc_lr = accuracy_score(y_val_lr, val_preds_lr)
# val_f1_lr = f1_score(y_val_lr, val_preds_lr, average="weighted")

# wandb.log({
#     "train_accuracy": train_acc_lr,
#     "train_f1": train_f1_lr,
#     "val_accuracy": val_acc_lr,
#     "val_f1": val_f1_lr
# })
# print(f"LR Val Accuracy: {val_acc_lr:.4f} | LR Val F1: {val_f1_lr:.4f}")

# wandb.finish()

In [ ]:
# X_test_lr = create_lr_features(test_df, w2v_model)
# lr_probs = lr_model.predict_proba(X_test_lr)
# lr_predictions = []
# for probs in lr_probs:
#     top3_idx = np.argsort(probs)[::-1][:3]
#     top3_labels = []
#     for idx in top3_idx:
#         letter_label = reverse_map[idx]
#         top3_labels.append(letter_label)
#     lr_predictions.append(" ".join(top3_labels))
# test_df["Prediction_Model_LR"] = lr_predictions

In [ ]:
final_submission = pd.DataFrame({
    "ID": test_df["id"],
    "Prediction": test_df["Prediction_Model_Rag"]})

final_submission.to_csv("submission.csv", index=False)
print(final_submission.head())